In [1]:
import sqlite3
import unicodedata
import pandas as pd

from dataclasses import dataclass
from pathlib import Path
from typing import Any

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

PROJECT_ROOT      = Path.cwd().parent
DB_PATH           = PROJECT_ROOT / "data" / "geonames.sqlite"
PACK_ID           = "latin"
GEONAMES_ISTANBUL = 745044

In [2]:
def get_conn() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn


def run_query(sql: str, params: tuple = ()) -> pd.DataFrame:
    with get_conn() as conn:
        return pd.read_sql_query(sql, conn, params=params)


def table_exists(name: str) -> bool:
    df = run_query(
        "SELECT 1 FROM sqlite_master WHERE type = 'table' AND name = ?",
        (name,),
    )
    return not df.empty


def normalise(text: str) -> str:
    """Mirrors compiler._normalise(): NFC + casefold + strip."""
    return unicodedata.normalize("NFC", text.strip()).casefold()

In [3]:
# All alternate names for Istanbul/Constantinople in GeoNames
run_query("""
SELECT
    isolanguage,
    alternate_name,
    is_preferred_name,
    is_historic,
    from_date,
    to_date,
    row_kind
FROM alternate_name
WHERE geonameid = ?
ORDER BY
    isolanguage,
    is_preferred_name DESC,
    is_historic DESC,
    alternate_name
""", (GEONAMES_ISTANBUL,))

,isolanguage,alternate_name,is_preferred_name,is_historic,from_date,to_date,row_kind
0,,Istanbul,1,0,None,None,name_untyped
1,,Byzantion,0,1,None,None,name_untyped
2,,Constantinopoli,0,1,None,None,name_untyped
3,,Constantinopolis,0,1,None,None,name_untyped
4,,Konstantinopel,0,1,None,None,name_untyped
5,,Konstantinoupolis,0,1,None,None,name_untyped
6,,Kustantiniyah,0,1,None,None,name_untyped
7,,Stamboul,0,1,None,None,name_untyped
8,,Stambul,0,1,None,None,name_untyped
9,,Κωνσταντινούπολις,0,1,None,None,name_untyped


In [4]:
# This is what the geonames_source pipeline step would find without the pack.
run_query("""
SELECT
    isolanguage,
    alternate_name,
    is_preferred_name,
    is_historic,
    from_date,
    to_date
FROM alternate_name
WHERE geonameid = ?
  AND isolanguage = 'la'
ORDER BY is_preferred_name DESC, alternate_name
""", (GEONAMES_ISTANBUL,))

,isolanguage,alternate_name,is_preferred_name,is_historic,from_date,to_date
0,la,Byzantium,0,1,None,None


In [5]:
# Check whether all pack tables are present

EXPECTED_TABLES = [
    "pack",
    "pack_install",
    "pack_pipeline_step",
    "pack_entity_name",
    "pack_entity_name_tag",
    "pack_string_exonym",
    "pack_import",
    "pack_row_provenance",
]

schema_df = pd.DataFrame([
    {"table": t, "exists": table_exists(t)}
    for t in EXPECTED_TABLES
])

assert schema_df["exists"].all(), (
    "Missing tables: "
    + ", ".join(schema_df.loc[~schema_df["exists"], "table"].tolist())
)

print("All expected tables present.")
schema_df

All expected tables present.


,table,exists
0,pack,True
1,pack_install,True
2,pack_pipeline_step,True
3,pack_entity_name,True
4,pack_entity_name_tag,True
5,pack_string_exonym,True
6,pack_import,True
7,pack_row_provenance,True


In [6]:
rows = []
for t in EXPECTED_TABLES:
    if table_exists(t):
        n = run_query(f"SELECT COUNT(*) AS n FROM {t}")["n"].iloc[0]
        rows.append({"table": t, "rows": int(n)})

pd.DataFrame(rows)

,table,rows
0,pack,1
1,pack_install,1
2,pack_pipeline_step,5
3,pack_entity_name,3
4,pack_entity_name_tag,6
5,pack_string_exonym,4
6,pack_import,2
7,pack_row_provenance,7


In [7]:
run_query("""
SELECT pack_id, display_name, bcp47, version, kind, default_mode, enabled
FROM pack
WHERE pack_id = ?
""", (PACK_ID,))

,pack_id,display_name,bcp47,version,kind,default_mode,enabled
0,latin,Latin,la,0.1.0,historical,entity_lookup,1


In [8]:
run_query("""
SELECT pack_id, source_path, content_hash, built_at
FROM pack_install
WHERE pack_id = ?
""", (PACK_ID,))

,pack_id,source_path,content_hash,built_at
0,latin,/home/max/Documents/Programming/NameTransductionEngine/packs/latin,f9775732117e3594f63a4038d4cf767cbe5038dc56d7cd1a9c70702a651e32c9,2026-04-12T12:59:31.399279+00:00


In [9]:
run_query("""
SELECT step_index, step_type, enabled, config_json
FROM pack_pipeline_step
WHERE pack_id = ?
ORDER BY step_index
""", (PACK_ID,))

,step_index,step_type,enabled,config_json
0,0,entity_lookup,1,"{""namespaces"": [""geonames"", ""wikidata""], ""step"": ""entity_lookup""}"
1,1,string_exonym,1,"{""step"": ""string_exonym""}"
2,2,geonames_source,1,"{""allow_untyped"": false, ""languages"": [""la""], ""merge"": true, ""step"": ""geonam..."
3,3,icu_transliteration,1,"{""step"": ""icu_transliteration"", ""transform"": ""Any-Latin; NFD; [:Mn:] Remove;..."
4,4,uroman,1,"{""step"": ""uroman""}"


In [10]:
run_query("""
SELECT
    namespace,
    entity_id,
    output_name,
    output_name_normal,
    valid_from,
    valid_to,
    priority,
    note
FROM pack_entity_name
WHERE pack_id = ?
ORDER BY namespace, entity_id, valid_from, priority DESC
""", (PACK_ID,))

,namespace,entity_id,output_name,output_name_normal,valid_from,valid_to,priority,note
0,geonames,745044,Byzantium,byzantium,-0660-01-01,0329-12-31,100,Latin name of the city during the pre-Constantinian era
1,geonames,745044,Constantinopolis,constantinopolis,0330-01-01,1453-12-31,100,Standard Latin name from Constantine's refoundation to the Ottoman conquest
2,geonames,745044,Nova Roma,nova roma,0330-01-01,0400-12-31,30,Ceremonial alternative used by Constantine himself; appears in top-k but not...


In [11]:
run_query("""
SELECT
    pen.namespace,
    pen.entity_id,
    pen.output_name,
    pen.valid_from,
    pen.valid_to,
    pen.priority,
    pent.tag
FROM pack_entity_name_tag AS pent
JOIN pack_entity_name AS pen
    ON  pen.pack_id    = pent.pack_id
    AND pen.namespace  = pent.namespace
    AND pen.entity_id  = pent.entity_id
    AND pen.output_name = pent.output_name
    AND pen.valid_from IS pent.valid_from
    AND pen.valid_to   IS pent.valid_to
WHERE pent.pack_id = ?
ORDER BY pen.namespace, pen.entity_id, pen.valid_from, pen.priority DESC, pent.tag
""", (PACK_ID,))

,namespace,entity_id,output_name,valid_from,valid_to,priority,tag
0,geonames,745044,Byzantium,-0660-01-01,0329-12-31,100,antique
1,geonames,745044,Byzantium,-0660-01-01,0329-12-31,100,historic
2,geonames,745044,Constantinopolis,0330-01-01,1453-12-31,100,historic
3,geonames,745044,Constantinopolis,0330-01-01,1453-12-31,100,medieval
4,geonames,745044,Nova Roma,0330-01-01,0400-12-31,30,ceremonial
5,geonames,745044,Nova Roma,0330-01-01,0400-12-31,30,historic


In [12]:
run_query("""
SELECT
    input_normal,
    output_name,
    output_name_normal,
    valid_from,
    valid_to,
    priority,
    note
FROM pack_string_exonym
WHERE pack_id = ?
ORDER BY input_normal, valid_from, priority DESC
""", (PACK_ID,))

,input_normal,output_name,output_name_normal,valid_from,valid_to,priority,note
0,constantinople,Constantinopolis,constantinopolis,0330-01-01,1453-12-31,100,Canonical Latinised form; higher-quality inputs deserve the same output
1,istanbul,Byzantium,byzantium,-0660-01-01,0329-12-31,100,String fallback when entity lookup is unavailable
2,istanbul,Constantinopolis,constantinopolis,0330-01-01,1453-12-31,100,String fallback when entity lookup is unavailable
3,new york,Novum Eboracum,novum eboracum,1500-01-01,2100-12-31,100,Neo-Latin humanist coinage; no GeoNames la-tagged alternate exists


In [13]:
run_query("""
SELECT
    pi.import_id,
    pi.pack_id,
    pi.source_file,
    pi.imported_at,
    COUNT(pr.row_pk_json) AS rows_imported
FROM pack_import AS pi
LEFT JOIN pack_row_provenance AS pr
    ON pr.import_id = pi.import_id
WHERE pi.pack_id = ?
GROUP BY pi.import_id
ORDER BY pi.import_id
""", (PACK_ID,))

,import_id,pack_id,source_file,imported_at,rows_imported
0,3,latin,entity_names.tsv,2026-04-12T12:59:31.399995+00:00,3
1,4,latin,string_exonyms.tsv,2026-04-12T12:59:31.401205+00:00,4


In [14]:
# Full provenance — every row with its source file and line number
run_query("""
SELECT
    pr.table_name,
    pr.row_pk_json,
    pi.source_file,
    pr.source_line
FROM pack_row_provenance AS pr
JOIN pack_import AS pi ON pi.import_id = pr.import_id
WHERE pr.pack_id = ?
ORDER BY pi.source_file, pr.source_line
""", (PACK_ID,))

,table_name,row_pk_json,source_file,source_line
0,pack_entity_name,"{""entity_id"": ""745044"", ""namespace"": ""geonames"", ""output_name"": ""Byzantium"",...",entity_names.tsv,2
1,pack_entity_name,"{""entity_id"": ""745044"", ""namespace"": ""geonames"", ""output_name"": ""Constantino...",entity_names.tsv,3
2,pack_entity_name,"{""entity_id"": ""745044"", ""namespace"": ""geonames"", ""output_name"": ""Nova Roma"",...",entity_names.tsv,4
3,pack_string_exonym,"{""input_normal"": ""istanbul"", ""output_name"": ""Byzantium"", ""pack_id"": ""latin"",...",string_exonyms.tsv,2
4,pack_string_exonym,"{""input_normal"": ""istanbul"", ""output_name"": ""Constantinopolis"", ""pack_id"": ""...",string_exonyms.tsv,3
5,pack_string_exonym,"{""input_normal"": ""constantinople"", ""output_name"": ""Constantinopolis"", ""pack_...",string_exonyms.tsv,4
6,pack_string_exonym,"{""input_normal"": ""new york"", ""output_name"": ""Novum Eboracum"", ""pack_id"": ""la...",string_exonyms.tsv,5


In [15]:
def resolve_entity(
    pack_id: str,
    namespace: str,
    entity_id: str,
    date: str,
    topk: int = 5,
) -> pd.DataFrame:
    """
    Mirrors the entity_lookup pipeline step: returns up to topk candidates
    from pack_entity_name whose validity range covers the given date,
    ordered by priority descending.

    Date comparison is text-order on (-)YYYY-MM-DD strings, which is correct
    for the sign-prefixed ISO 8601 format used in this schema.
    """
    return run_query(
        """
        SELECT
            output_name,
            valid_from,
            valid_to,
            priority,
            note
        FROM pack_entity_name
        WHERE pack_id   = ?
          AND namespace = ?
          AND entity_id = ?
          AND (valid_from IS NULL OR valid_from <= ?)
          AND (valid_to   IS NULL OR valid_to   >= ?)
        ORDER BY priority DESC, valid_from DESC
        LIMIT ?
        """,
        (pack_id, namespace, str(entity_id), date, date, topk),
    )


def resolve_string(
    pack_id: str,
    input_str: str,
    date: str,
    topk: int = 5,
) -> pd.DataFrame:
    """
    Mirrors the string_exonym pipeline step: normalises the input string
    and returns up to topk candidates from pack_string_exonym.
    """
    return run_query(
        """
        SELECT
            input_normal,
            output_name,
            valid_from,
            valid_to,
            priority,
            note
        FROM pack_string_exonym
        WHERE pack_id      = ?
          AND input_normal = ?
          AND (valid_from IS NULL OR valid_from <= ?)
          AND (valid_to   IS NULL OR valid_to   >= ?)
        ORDER BY priority DESC, valid_from DESC
        LIMIT ?
        """,
        (pack_id, normalise(input_str), date, date, topk),
    )


def resolve_geonames(
    geonameid: int,
    date: str,
    languages: tuple[str, ...] = ("la",),
    topk: int = 5,
) -> pd.DataFrame:
    """
    Mirrors the geonames_source pipeline step: queries the alternate_name
    table for the given languages, preferring rows whose from_date/to_date
    bracket the query date when available.
    """
    lang_placeholders = ",".join("?" for _ in languages)
    return run_query(
        f"""
        SELECT
            isolanguage,
            alternate_name,
            is_preferred_name,
            is_historic,
            from_date,
            to_date,
            row_kind
        FROM alternate_name
        WHERE geonameid   = ?
          AND isolanguage IN ({lang_placeholders})
        ORDER BY
            is_preferred_name DESC,
            CASE
                WHEN from_date IS NOT NULL AND to_date IS NOT NULL
                 AND from_date <= ? AND to_date >= ?
                THEN 1 ELSE 0
            END DESC,
            is_historic DESC,
            alternate_name
        LIMIT ?
        """,
        (geonameid, *languages, date, date, topk),
    )


In [16]:
@dataclass
class Result:
    test_id:  str
    desc:     str
    passed:   bool
    actual:   Any
    expected: Any
    note:     str = ""


_results: list[Result] = []


def _record(r: Result) -> Result:
    _results.append(r)
    icon = "✅" if r.passed else "❌"
    status = "PASS" if r.passed else "FAIL"
    print(f"{icon} [{status}] {r.test_id}: {r.desc}")
    if not r.passed:
        print(f"         expected: {r.expected!r}")
        print(f"         actual:   {r.actual!r}")
    return r


def check_entity_top1(
    test_id: str,
    desc: str,
    pack_id: str,
    namespace: str,
    entity_id: Any,
    date: str,
    expected_top1: str,
) -> Result:
    df = resolve_entity(pack_id, namespace, str(entity_id), date, topk=1)
    actual = df["output_name"].iloc[0] if not df.empty else None
    return _record(Result(test_id, desc, actual == expected_top1, actual, expected_top1))


def check_entity_contains(
    test_id: str,
    desc: str,
    pack_id: str,
    namespace: str,
    entity_id: Any,
    date: str,
    must_contain: list[str],
    topk: int = 5,
) -> Result:
    df = resolve_entity(pack_id, namespace, str(entity_id), date, topk=topk)
    found = set(df["output_name"].tolist())
    missing = [n for n in must_contain if n not in found]
    passed = len(missing) == 0
    actual = sorted(found)
    expected = f"contains {must_contain}"
    return _record(Result(test_id, desc, passed, actual, expected,
                          note=f"missing: {missing}" if missing else ""))


def check_entity_not_contains(
    test_id: str,
    desc: str,
    pack_id: str,
    namespace: str,
    entity_id: Any,
    date: str,
    must_not_contain: list[str],
    topk: int = 5,
) -> Result:
    df = resolve_entity(pack_id, namespace, str(entity_id), date, topk=topk)
    found = set(df["output_name"].tolist())
    unexpected = [n for n in must_not_contain if n in found]
    passed = len(unexpected) == 0
    actual = sorted(found)
    expected = f"not contains {must_not_contain}"
    return _record(Result(test_id, desc, passed, actual, expected,
                          note=f"unexpectedly present: {unexpected}" if unexpected else ""))


def check_string_top1(
    test_id: str,
    desc: str,
    pack_id: str,
    input_str: str,
    date: str,
    expected_top1: str,
) -> Result:
    df = resolve_string(pack_id, input_str, date, topk=1)
    actual = df["output_name"].iloc[0] if not df.empty else None
    return _record(Result(test_id, desc, actual == expected_top1, actual, expected_top1))


def check_fallback(
    test_id: str,
    desc: str,
    pack_id: str,
    namespace: str,
    entity_id: Any,
    input_str: str,
    date: str,
) -> Result:
    """
    Asserts that both entity_lookup and string_exonym return no candidates,
    meaning the engine must fall through to the transliteration tier.
    """
    entity_df = resolve_entity(pack_id, namespace, str(entity_id), date)
    string_df  = resolve_string(pack_id, input_str, date)
    both_empty = entity_df.empty and string_df.empty
    actual   = {"entity_empty": entity_df.empty, "string_empty": string_df.empty}
    expected = "both lookup steps return no candidates"
    return _record(Result(test_id, desc, both_empty, actual, expected))


def check_normalisation(
    test_id: str,
    desc: str,
    pack_id: str,
    namespace: str,
    entity_id: Any,
    input_variants: list[str],
    date: str,
) -> Result:
    """
    Asserts that all input variants resolve to the same top-1 entity name.
    """
    results = {}
    for v in input_variants:
        df = resolve_entity(pack_id, namespace, str(entity_id), date, topk=1)
        results[v] = df["output_name"].iloc[0] if not df.empty else None
    top1_values = set(results.values())
    passed = len(top1_values) == 1 and None not in top1_values
    return _record(Result(test_id, desc, passed, results,
                          f"all variants resolve to same name"))


In [17]:
# istanbul_antique: antique era → Byzantium
check_entity_top1(
    "istanbul_antique",
    "Istanbul at date 0140-01-01 resolves to Byzantium",
    PACK_ID, "geonames", GEONAMES_ISTANBUL,
    date="0140-01-01",
    expected_top1="Byzantium",
)

# istanbul_medieval: medieval era → Constantinopolis
check_entity_top1(
    "istanbul_medieval",
    "Istanbul at date 1100-01-01 resolves to Constantinopolis",
    PACK_ID, "geonames", GEONAMES_ISTANBUL,
    date="1100-01-01",
    expected_top1="Constantinopolis",
)

# constantinople_medieval: different input form, same entity → same result
check_string_top1(
    "constantinople_medieval",
    "String 'Constantinople' at 1200-01-01 resolves to Constantinopolis",
    PACK_ID,
    input_str="Constantinople",
    date="1200-01-01",
    expected_top1="Constantinopolis",
)


✅ [PASS] istanbul_antique: Istanbul at date 0140-01-01 resolves to Byzantium
✅ [PASS] istanbul_medieval: Istanbul at date 1100-01-01 resolves to Constantinopolis
✅ [PASS] constantinople_medieval: String 'Constantinople' at 1200-01-01 resolves to Constantinopolis


Result(test_id='constantinople_medieval', desc="String 'Constantinople' at 1200-01-01 resolves to Constantinopolis", passed=True, actual='Constantinopolis', expected='Constantinopolis', note='')

In [18]:
# Last day of Byzantium validity
check_entity_top1(
    "boundary_last_antique_day",
    "0329-12-31 is the last valid day for Byzantium",
    PACK_ID, "geonames", GEONAMES_ISTANBUL,
    date="0329-12-31",
    expected_top1="Byzantium",
)

# First day of Constantinopolis validity
check_entity_top1(
    "boundary_first_medieval_day",
    "0330-01-01 is the first valid day for Constantinopolis",
    PACK_ID, "geonames", GEONAMES_ISTANBUL,
    date="0330-01-01",
    expected_top1="Constantinopolis",
)

# Last day of Constantinopolis validity
check_entity_top1(
    "boundary_last_constantinopolis_day",
    "1453-12-31 is the last valid day for Constantinopolis",
    PACK_ID, "geonames", GEONAMES_ISTANBUL,
    date="1453-12-31",
    expected_top1="Constantinopolis",
)


✅ [PASS] boundary_last_antique_day: 0329-12-31 is the last valid day for Byzantium
✅ [PASS] boundary_first_medieval_day: 0330-01-01 is the first valid day for Constantinopolis
✅ [PASS] boundary_last_constantinopolis_day: 1453-12-31 is the last valid day for Constantinopolis


Result(test_id='boundary_last_constantinopolis_day', desc='1453-12-31 is the last valid day for Constantinopolis', passed=True, actual='Constantinopolis', expected='Constantinopolis', note='')

In [19]:
check_fallback(
    "boundary_post_ottoman",
    "1454-01-01: no Latin entity or string entry — engine falls to transliteration",
    PACK_ID, "geonames", GEONAMES_ISTANBUL,
    input_str="Istanbul",
    date="1454-01-01",
)

✅ [PASS] boundary_post_ottoman: 1454-01-01: no Latin entity or string entry — engine falls to transliteration


Result(test_id='boundary_post_ottoman', desc='1454-01-01: no Latin entity or string entry — engine falls to transliteration', passed=True, actual={'entity_empty': True, 'string_empty': True}, expected='both lookup steps return no candidates', note='')

In [20]:
# Confirm manually what the two lookup steps return
print("Entity lookup (geonameid=745044, 1454-01-01):")
display(resolve_entity(PACK_ID, "geonames", str(GEONAMES_ISTANBUL), "1454-01-01"))

print("String exonym (Istanbul, 1454-01-01):")
display(resolve_string(PACK_ID, "Istanbul", "1454-01-01"))

Entity lookup (geonameid=745044, 1454-01-01):


,output_name,valid_from,valid_to,priority,note


String exonym (Istanbul, 1454-01-01):


,input_normal,output_name,valid_from,valid_to,priority,note


In [21]:
# Nova Roma present during its validity window
check_entity_contains(
    "nova_roma_in_top_k",
    "Nova Roma appears as a candidate at date 0340-01-01",
    PACK_ID, "geonames", GEONAMES_ISTANBUL,
    date="0340-01-01",
    must_contain=["Nova Roma"],
    topk=5,
)

# Nova Roma absent after its valid_to
check_entity_not_contains(
    "nova_roma_absent_after_validity",
    "Nova Roma must not appear after 0400-12-31",
    PACK_ID, "geonames", GEONAMES_ISTANBUL,
    date="0401-01-01",
    must_not_contain=["Nova Roma"],
    topk=5,
)

✅ [PASS] nova_roma_in_top_k: Nova Roma appears as a candidate at date 0340-01-01
✅ [PASS] nova_roma_absent_after_validity: Nova Roma must not appear after 0400-12-31


Result(test_id='nova_roma_absent_after_validity', desc='Nova Roma must not appear after 0400-12-31', passed=True, actual=['Constantinopolis'], expected="not contains ['Nova Roma']", note='')

In [22]:
# Show the full top-5 candidate set for 0340-01-01 for inspection
print("Top-5 entity candidates for geonameid=745044 at 0340-01-01:")
resolve_entity(PACK_ID, "geonames", str(GEONAMES_ISTANBUL), "0340-01-01", topk=5)

Top-5 entity candidates for geonameid=745044 at 0340-01-01:


,output_name,valid_from,valid_to,priority,note
0,Constantinopolis,0330-01-01,1453-12-31,100,Standard Latin name from Constantine's refoundation to the Ottoman conquest
1,Nova Roma,0330-01-01,0400-12-31,30,Ceremonial alternative used by Constantine himself; appears in top-k but not...


In [23]:
# Novum Eboracum resolves in the Neo-Latin era
check_string_top1(
    "novum_eboracum",
    "New York at 1700-01-01 resolves to Novum Eboracum via string_exonym",
    PACK_ID,
    input_str="New York",
    date="1700-01-01",
    expected_top1="Novum Eboracum",
)

# Novum Eboracum is not valid before the Neo-Latin era (valid_from=1500-01-01)
check_fallback(
    "novum_eboracum_not_before_era",
    "New York at 1200-01-01 has no string entry — falls to transliteration",
    PACK_ID, "geonames", -1,   # no geonameid for New York in this pack
    input_str="New York",
    date="1200-01-01",
)


✅ [PASS] novum_eboracum: New York at 1700-01-01 resolves to Novum Eboracum via string_exonym
✅ [PASS] novum_eboracum_not_before_era: New York at 1200-01-01 has no string entry — falls to transliteration


Result(test_id='novum_eboracum_not_before_era', desc='New York at 1200-01-01 has no string entry — falls to transliteration', passed=True, actual={'entity_empty': True, 'string_empty': True}, expected='both lookup steps return no candidates', note='')

In [24]:
# Show what string lookup returns for New York across both dates
print("String lookup: New York at 1700-01-01")
display(resolve_string(PACK_ID, "New York", "1700-01-01"))

print("String lookup: New York at 1200-01-01")
display(resolve_string(PACK_ID, "New York", "1200-01-01"))

String lookup: New York at 1700-01-01


,input_normal,output_name,valid_from,valid_to,priority,note
0,new york,Novum Eboracum,1500-01-01,2100-12-31,100,Neo-Latin humanist coinage; no GeoNames la-tagged alternate exists


String lookup: New York at 1200-01-01


,input_normal,output_name,valid_from,valid_to,priority,note


In [25]:
check_fallback(
    "unknown_place_fallback",
    "Timbuktu has no pack entry at any date — falls to transliteration",
    PACK_ID, "geonames", -1,
    input_str="Timbuktu",
    date="1300-01-01",
)

✅ [PASS] unknown_place_fallback: Timbuktu has no pack entry at any date — falls to transliteration


Result(test_id='unknown_place_fallback', desc='Timbuktu has no pack entry at any date — falls to transliteration', passed=True, actual={'entity_empty': True, 'string_empty': True}, expected='both lookup steps return no candidates', note='')

In [26]:
# Entity lookup is keyed by entity_id (a GeoNames integer), so it is
# entirely immune to input capitalisation — the input string is only used
# by the string_exonym step.  Verify that string normalisation works there.
variants = ["Istanbul", "ISTANBUL", "istanbul", "Istanbul"]
rows = []
for v in variants:
    df = resolve_string(PACK_ID, v, "0140-01-01", topk=1)
    top1 = df["output_name"].iloc[0] if not df.empty else None
    rows.append({"input": v, "normalised": normalise(v), "top1": top1})

result_df = pd.DataFrame(rows)
display(result_df)

# All variants must produce the same top-1 result
top1_values = set(result_df["top1"].dropna())
passed = len(top1_values) == 1
_record(Result(
    "input_normalisation",
    "All capitalisation variants of Istanbul resolve to the same output",
    passed,
    actual=result_df["top1"].tolist(),
    expected="all identical",
))


,input,normalised,top1
0,Istanbul,istanbul,Byzantium
1,ISTANBUL,istanbul,Byzantium
2,istanbul,istanbul,Byzantium
3,Istanbul,istanbul,Byzantium


✅ [PASS] input_normalisation: All capitalisation variants of Istanbul resolve to the same output


Result(test_id='input_normalisation', desc='All capitalisation variants of Istanbul resolve to the same output', passed=True, actual=['Byzantium', 'Byzantium', 'Byzantium', 'Byzantium'], expected='all identical', note='')

In [27]:
# Entity lookup is keyed by entity_id (a GeoNames integer), so it is
# entirely immune to input capitalisation — the input string is only used
# by the string_exonym step.  Verify that string normalisation works there.
variants = ["Istanbul", "ISTANBUL", "istanbul", "Istanbul"]
rows = []
for v in variants:
    df = resolve_string(PACK_ID, v, "0140-01-01", topk=1)
    top1 = df["output_name"].iloc[0] if not df.empty else None
    rows.append({"input": v, "normalised": normalise(v), "top1": top1})

result_df = pd.DataFrame(rows)
display(result_df)

# All variants must produce the same top-1 result
top1_values = set(result_df["top1"].dropna())
passed = len(top1_values) == 1
_record(Result(
    "input_normalisation",
    "All capitalisation variants of Istanbul resolve to the same output",
    passed,
    actual=result_df["top1"].tolist(),
    expected="all identical",
))


,input,normalised,top1
0,Istanbul,istanbul,Byzantium
1,ISTANBUL,istanbul,Byzantium
2,istanbul,istanbul,Byzantium
3,Istanbul,istanbul,Byzantium


✅ [PASS] input_normalisation: All capitalisation variants of Istanbul resolve to the same output


Result(test_id='input_normalisation', desc='All capitalisation variants of Istanbul resolve to the same output', passed=True, actual=['Byzantium', 'Byzantium', 'Byzantium', 'Byzantium'], expected='all identical', note='')

In [28]:
print("GeoNames la-tagged candidates at 0140-01-01 (antique era):")
display(resolve_geonames(GEONAMES_ISTANBUL, "0140-01-01", languages=("la",)))

print("GeoNames la-tagged candidates at 1100-01-01 (medieval era):")
display(resolve_geonames(GEONAMES_ISTANBUL, "1100-01-01", languages=("la",)))

GeoNames la-tagged candidates at 0140-01-01 (antique era):


,isolanguage,alternate_name,is_preferred_name,is_historic,from_date,to_date,row_kind
0,la,Byzantium,0,1,None,None,name_lang


GeoNames la-tagged candidates at 1100-01-01 (medieval era):


,isolanguage,alternate_name,is_preferred_name,is_historic,from_date,to_date,row_kind
0,la,Byzantium,0,1,None,None,name_lang


In [29]:
summary = pd.DataFrame([
    {
        "id":      r.test_id,
        "desc":    r.desc,
        "result":  "PASS" if r.passed else "FAIL",
        "actual":  str(r.actual),
        "expected": str(r.expected),
        "note":    r.note,
    }
    for r in _results
])

passed = summary["result"].eq("PASS").sum()
failed = summary["result"].eq("FAIL").sum()
total  = len(summary)

print(f"Results: {passed}/{total} passed, {failed} failed")
if failed:
    print("Failing cases:")
    print(summary.loc[summary["result"] == "FAIL", ["id", "desc", "actual", "expected", "note"]].to_string(index=False))

summary[["id", "result", "desc", "note"]]


Results: 14/14 passed, 0 failed


,id,result,desc,note
0,istanbul_antique,PASS,Istanbul at date 0140-01-01 resolves to Byzantium,
1,istanbul_medieval,PASS,Istanbul at date 1100-01-01 resolves to Constantinopolis,
2,constantinople_medieval,PASS,String 'Constantinople' at 1200-01-01 resolves to Constantinopolis,
3,boundary_last_antique_day,PASS,0329-12-31 is the last valid day for Byzantium,
4,boundary_first_medieval_day,PASS,0330-01-01 is the first valid day for Constantinopolis,
5,boundary_last_constantinopolis_day,PASS,1453-12-31 is the last valid day for Constantinopolis,
6,boundary_post_ottoman,PASS,1454-01-01: no Latin entity or string entry — engine falls to transliteration,
7,nova_roma_in_top_k,PASS,Nova Roma appears as a candidate at date 0340-01-01,
8,nova_roma_absent_after_validity,PASS,Nova Roma must not appear after 0400-12-31,
9,novum_eboracum,PASS,New York at 1700-01-01 resolves to Novum Eboracum via string_exonym,
